# Letterboxd Popular Films: Overview & Quality Audit
This notebook connects to the raw SQLite database, verifies the overall entity counts, and checks the data completeness based on the Data Card specifications.

In [1]:
import sqlite3
import pandas as pd

db_path = '../data/raw/letterboxd.db'
conn = sqlite3.connect(db_path)

print("Database connection established.")

Database connection established.


In [2]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)

print("Tables in the database:")
print(tables)

Tables in the database:
              name
0            films
1  sqlite_sequence
2     cast_members
3     crew_members
4     film_details
5      film_genres
6     film_reviews


In [3]:
# 2. Verify Key Statistics (Entity Counts)
queries_counts = {
    "Films": "SELECT COUNT(*) FROM films;",
    "Cast Members": "SELECT COUNT(*) FROM cast_members;",
    "Unique Cast Members": "SELECT COUNT(DISTINCT actor_name) FROM cast_members;",
    "Crew Members": "SELECT COUNT(*) FROM crew_members;",
    "Unique Crew Members": "SELECT COUNT(DISTINCT person_name) FROM crew_members;",
    "Film Details": "SELECT COUNT(*) FROM film_details;",
    "Film Genres": "SELECT COUNT(*) FROM film_genres;",
    "Unique Genres": "SELECT COUNT(DISTINCT genre_name) FROM film_genres;",
    "Film Reviews": "SELECT COUNT(*) FROM film_reviews;"
}

counts_data = []
for entity, query in queries_counts.items():
    count = pd.read_sql(query, conn).iloc[0, 0]
    counts_data.append({"Entity": entity, "Count": count})

df_counts = pd.DataFrame(counts_data)
display(df_counts)

,Entity,Count
0,Films,1754
1,Cast Members,134753
2,Unique Cast Members,63636
3,Crew Members,123539
4,Unique Crew Members,50289
5,Film Details,11818
6,Film Genres,4666
7,Unique Genres,22
8,Film Reviews,175400


In [4]:
# 3. Data Quality & Completeness Checks
quality_checks = {
    "Missing Taglines": "SELECT COUNT(*) FROM films WHERE tagline IS NULL OR tagline = '';",
    "Missing Synopsis": "SELECT COUNT(*) FROM films WHERE synopsis IS NULL OR synopsis = '';",
    "Missing Review Ratings": "SELECT COUNT(*) FROM film_reviews WHERE star_rating IS NULL OR star_rating = '';",
    "Missing Review Text": "SELECT COUNT(*) FROM film_reviews WHERE review_text IS NULL OR review_text = '';"
}

quality_data = []
for metric, query in quality_checks.items():
    count = pd.read_sql(query, conn).iloc[0, 0]
    quality_data.append({"Metric": metric, "Count": count})

df_quality = pd.DataFrame(quality_data)
display(df_quality)

,Metric,Count
0,Missing Taglines,81
1,Missing Synopsis,0
2,Missing Review Ratings,9207
3,Missing Review Text,4


# Demographics & Distribution Profiles
This notebook visualizes the distribution of overall ratings, release years, and extracts the top countries, languages, and genres.